In [2]:
import pandas as pd
matches_df = pd.read_csv('matches.csv')
deliveries_df = pd.read_csv('deliveries.csv')

In [3]:
print("Matches DataFrame:")
print(matches_df.head())

print("\nDeliveries DataFrame:")
print(deliveries_df.head())

Matches DataFrame:
       id   season        city        date match_type player_of_match  \
0  335982  2007/08   Bangalore  2008-04-18     League     BB McCullum   
1  335983  2007/08  Chandigarh  2008-04-19     League      MEK Hussey   
2  335984  2007/08       Delhi  2008-04-19     League     MF Maharoof   
3  335985  2007/08      Mumbai  2008-04-20     League      MV Boucher   
4  335986  2007/08     Kolkata  2008-04-20     League       DJ Hussey   

                                        venue                        team1  \
0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
2                            Feroz Shah Kotla             Delhi Daredevils   
3                            Wankhede Stadium               Mumbai Indians   
4                                Eden Gardens        Kolkata Knight Riders   

                         team2                  toss_winner toss_decision

In [11]:
if 'city' in matches_df.columns:

    matches_df = matches_df.rename(columns={'city': 'City'})
elif 'City' not in matches_df.columns:

    matches_df['City'] = 'Unknown_City'

matches_df['City'] = matches_df['City'].fillna('Unknown_City')

if 'Venue' in matches_df.columns:
    matches_df['Venue'] = matches_df['Venue'].fillna('Unknown_Venue')
else:
    print("Warning: 'Venue' column not found in the DataFrame.")

In [12]:
if 'City' in deliveries_df.columns:
    deliveries_df['City'] = deliveries_df['City'].fillna('Unknown_City')
else:
    print("Warning: 'City' column not found in deliveries DataFrame.")

In [6]:
if 'total_run' in deliveries_df.columns:
    deliveries_df['total_run'] = deliveries_df['total_run'].fillna(deliveries_df['total_run'].mean())

In [7]:
print("\nMatches DataFrame Info After Handling Missing Values:")
print(matches_df.info())

print("\nDeliveries DataFrame Info After Handling Missing Values:")
print(deliveries_df.info())


Matches DataFrame Info After Handling Missing Values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   object 
 2   City             1095 non-null   object 
 3   date             1095 non-null   object 
 4   match_type       1095 non-null   object 
 5   player_of_match  1090 non-null   object 
 6   venue            1095 non-null   object 
 7   team1            1095 non-null   object 
 8   team2            1095 non-null   object 
 9   toss_winner      1095 non-null   object 
 10  toss_decision    1095 non-null   object 
 11  winner           1090 non-null   object 
 12  result           1095 non-null   object 
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super

In [13]:
merged_df = pd.merge(deliveries_df, matches_df, left_on='match_id', right_on='id', how='left')
print("\nMerged DataFrame:")
print(merged_df.head())


Merged DataFrame:
   match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
4    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball       batter   bowler  non_striker  batsman_runs  ...  toss_decision  \
0     1   SC Ganguly  P Kumar  BB McCullum           0.0  ...          field   
1     2  BB McCullum  P Kumar   SC Ganguly           0.0  ...          field   
2     3  BB McCullum  P Kumar   SC Ganguly           0.0  ...          field   
3     4  BB McCullum  P Kumar   SC Ganguly           0.0  ...          field   
4     5  BB McCullum  P Kumar   SC Ganguly           0.0  ...          field   

             

In [14]:
merged_df['team1'] = merged_df['team1']
merged_df['team2'] = merged_df['team2']
print("\nRenamed Teams:")
print(merged_df[['team1', 'team2']].head())


Renamed Teams:
                         team1                  team2
0  Royal Challengers Bangalore  Kolkata Knight Riders
1  Royal Challengers Bangalore  Kolkata Knight Riders
2  Royal Challengers Bangalore  Kolkata Knight Riders
3  Royal Challengers Bangalore  Kolkata Knight Riders
4  Royal Challengers Bangalore  Kolkata Knight Riders


In [15]:
merged_df = pd.merge(deliveries_df, matches_df, left_on='match_id', right_on='id', how='left')
if 'total_run' in deliveries_df.columns:
    merged_df['total_run'] = deliveries_df['total_run']
else:
    merged_df['total_run'] = 0

merged_df = merged_df[['match_id', 'inning', 'total_run', *[col for col in merged_df.columns if col not in ['match_id', 'inning', 'total_run']]]]

merged_df['Cumulative_Runs'] = merged_df.groupby(['match_id', 'inning'])['total_run'].cumsum()
print("\nCumulative Runs:")
print(merged_df[['match_id', 'inning', 'Cumulative_Runs']].head())


Cumulative Runs:
   match_id  inning  Cumulative_Runs
0    335982       1                0
1    335982       1                0
2    335982       1                0
3    335982       1                0
4    335982       1                0


In [16]:
merged_df['Cumulative_Wickets'] = merged_df.groupby(['match_id', 'inning'])['player_dismissed'].transform(lambda x: x.notna().cumsum())
print("\nCumulative Wickets:")
print(merged_df[['match_id', 'inning', 'Cumulative_Wickets']].head())


Cumulative Wickets:
   match_id  inning  Cumulative_Wickets
0    335982       1                   0
1    335982       1                   0
2    335982       1                   0
3    335982       1                   0
4    335982       1                   0


In [17]:
merged_df['Current_Run_Rate'] = merged_df.groupby(['match_id', 'inning'])['total_run'].cumsum() / (merged_df.groupby(['match_id', 'inning'])['over'].cummax() + 0.1)
print("\nCurrent Run Rate:")
print(merged_df[['match_id', 'inning', 'Current_Run_Rate']].head())


Current Run Rate:
   match_id  inning  Current_Run_Rate
0    335982       1               0.0
1    335982       1               0.0
2    335982       1               0.0
3    335982       1               0.0
4    335982       1               0.0


In [18]:
merged_df['Overs_Completed'] = merged_df.groupby(['match_id', 'inning'])['over'].cummax()
print("\nOvers Completed:")
print(merged_df[['match_id', 'inning', 'Overs_Completed']].head())


Overs Completed:
   match_id  inning  Overs_Completed
0    335982       1                0
1    335982       1                0
2    335982       1                0
3    335982       1                0
4    335982       1                0


In [19]:
merged_df.to_csv('enhanced_ipl_data_with_new_features.csv', index=False)
print("\nEnhanced dataset saved to 'enhanced_ipl_data_with_new_features.csv'")


Enhanced dataset saved to 'enhanced_ipl_data_with_new_features.csv'


In [20]:
import pandas as pd

enhanced_df = pd.read_csv('enhanced_ipl_data_with_new_features.csv')

enhanced_df.head()

<ipython-input-20-415b98f1741c>:3: DtypeWarning: Columns (19,35) have mixed types. Specify dtype option on import or set low_memory=False.
  enhanced_df = pd.read_csv('enhanced_ipl_data_with_new_features.csv')


,match_id,inning,total_run,batting_team,bowling_team,over,ball,batter,bowler,non_striker,...,target_runs,target_overs,super_over,method,umpire1,umpire2,Cumulative_Runs,Cumulative_Wickets,Current_Run_Rate,Overs_Completed
0,335982,1,0,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,...,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen,0,0,0.0,0
1,335982,1,0,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,...,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen,0,0,0.0,0
2,335982,1,0,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,...,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen,0,0,0.0,0
3,335982,1,0,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,...,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen,0,0,0.0,0
4,335982,1,0,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,...,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen,0,0,0.0,0


In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
merged_df = pd.read_csv('enhanced_ipl_data_with_new_features.csv')

<ipython-input-21-b07203fb8116>:3: DtypeWarning: Columns (19,35) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_df = pd.read_csv('enhanced_ipl_data_with_new_features.csv')


In [22]:
merged_df.fillna({'City': 'Unknown', 'Venue': 'Unknown'}, inplace=True)
if 'Venue' not in merged_df.columns:
    merged_df['Venue'] = 'Unknown'

In [23]:
merged_df['win'] = (merged_df['batting_team'] == merged_df['winner']).astype(int)
merged_df.drop('winner', axis=1, inplace=True)

In [24]:
merged_encoded = pd.get_dummies(
    merged_df,
    columns=['City', 'Venue', 'toss_decision', 'batting_team', 'bowling_team'],
    drop_first=True
)

Number of columns after encoding

In [26]:
print(f"Number of columns after encoding: {merged_encoded.shape[1]}")

Number of columns after encoding: 95


In [25]:
X = merged_encoded.drop('win', axis=1)
y = merged_encoded['win']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

Training set: 130289 samples
Test set: 32573 samples
